# Carbon Nanotube Band Structures

Este cuaderno construye geometrías representativas de nanotubos de carbono y calcula sus estructuras de bandas utilizando un modelo Tight-Binding de primer vecino.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

# Garantizar que el paquete `src` esté en el path cuando el cuaderno se ejecute
project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root / "src"))

from cnt_generator import (
    armchair_cnt,
    zigzag_cnt,
    chiral_parameters,
    graphene_nearest_neighbor_vectors,
)


In [ ]:
GAMMA0 = -3.0  # eV


def band_structure_data(n, m, *, num_k=601, gamma0=GAMMA0):
    """Devuelve el eje de k, los autovalores y el gap para un CNT (n, m)."""

    params = chiral_parameters(n, m)
    reciprocal_basis = params["reciprocal_basis"]
    k1 = reciprocal_basis[:, 0]
    k2 = reciprocal_basis[:, 1]
    deltas = graphene_nearest_neighbor_vectors()

    lambda_vals = np.linspace(-0.5, 0.5, num_k)
    k_parallel = lambda_vals * np.linalg.norm(k2)
    num_subbands = params["num_subbands"]

    energies = np.zeros((2 * num_subbands, num_k), dtype=float)
    for q_idx, q in enumerate(range(num_subbands)):
        k_offset = q * k1
        for lam_idx, lam in enumerate(lambda_vals):
            k_vec = k_offset + lam * k2
            phase = np.array([np.dot(k_vec, delta[:2]) for delta in deltas])
            hopping_sum = np.sum(np.exp(1j * phase))
            off_diag = gamma0 * hopping_sum
            h_k = np.array([[0.0, off_diag], [np.conj(off_diag), 0.0]], dtype=complex)
            energies[2 * q_idx : 2 * q_idx + 2, lam_idx] = np.sort(np.linalg.eigvalsh(h_k))

    positives = energies[energies > 0.0]
    negatives = energies[energies < 0.0]
    if positives.size and negatives.size:
        band_gap = float(positives.min() - negatives.max())
    else:
        band_gap = 0.0

    return k_parallel, energies, band_gap, params


def band_figure(k_parallel, energies, title):
    fig = go.Figure()
    for band in energies:
        fig.add_trace(
            go.Scatter(
                x=k_parallel,
                y=band,
                mode="lines",
                line=dict(width=1.0, color="#1f77b4"),
                hoverinfo="skip",
                showlegend=False,
            )
        )
    fig.add_hline(y=0.0, line=dict(color="black", dash="dash"))
    fig.update_layout(
        title=title,
        template="plotly_white",
        xaxis=dict(title=r"$k_\\parallel$ (1/Å)", zeroline=False),
        yaxis=dict(title="Energía (eV)", zeroline=False),
        margin=dict(l=60, r=30, t=60, b=60),
    )
    return fig


In [ ]:
armchair_geom = armchair_cnt(length_repeats=4)
zigzag_geom = zigzag_cnt(length_repeats=4)

armchair_data = band_structure_data(5, 5)
zigzag_data = band_structure_data(9, 0)

for label, geom, data in [
    ("CNT (5,5) - Armchair", armchair_geom, armchair_data),
    ("CNT (9,0) - Zig-Zag", zigzag_geom, zigzag_data),
]:
    k_axis, energies, band_gap, params = data
    print(f"{label}")
    print(f"  Átomos en celda repetida: {len(geom.positions)}")
    print(f"  Subbandas (N): {params['num_subbands']}")
    print(f"  Gap electrónico: {band_gap:.4f} eV\n")


In [ ]:
fig_armchair = band_figure(
    armchair_data[0], armchair_data[1], "Estructura de bandas CNT Armchair (5,5)"
)
fig_armchair


In [ ]:
fig_zigzag = band_figure(
    zigzag_data[0], zigzag_data[1], "Estructura de bandas CNT Zig-Zag (9,0)"
)
fig_zigzag
